<a href="https://colab.research.google.com/github/RezZup/MixBhasha_git/blob/main/notebooks/MixBhasha_V5_Train_Colab_Kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗣️ MixBhasha v5 — benchmark-aware code-mixed NLP

**Training targets:** Hinglish sentiment (3-way, processed or raw-text team mirror), MC-Hinglish cyberbullying (7-way), derived broad cyberbullying flag (2-way). An authorized HASOC HOF/NOT task and separate PRISM hate task are optional, **never silently merged**.

### What is different from v4?
- Uses **all seven MC-Hinglish1.0 labels**; the derived binary task is clearly called *cyberbullying-category vs not-cyberbullying*, **not a universal abuse detector**.
- Adds **HingRoBERTa-Mixed**, **Twitter-XLM-R**, **Twitter-XLM-R sentiment-transfer**, and calibrated **TF-IDF+LinearSVM** baselines.
- Replaces label-dependent cross-split quarantine with **training-label-only conflict removal**; removes exact train/held-out text collisions without reading held-out labels to make that decision.
- Stores validation probability vectors for **paired bootstrap comparisons**, and optionally tests a **validation-only ensemble**.
- Saves split fingerprints, per-class metrics, checkpoint state, script robustness, and independent human-annotated challenge evaluation.

**Important:** SentiMix public development slice was evaluated in v4. Re-evaluation in v5 is *historical / exploratory*, not an untouched independent final test. Do **not** claim SemEval leaderboard superiority. The new challenge set must be independently annotated. No model scores are prefilled.

**GPU recommendations:** Colab T4 / Kaggle T4 with 16GB RAM; enable internet for model and dataset downloads. Keep only **1 transformer per session** on the free tier.

In [1]:
# Avoid upgrading torch or installing tensorflow; use the environment's CUDA-compatible torch.
%pip -q install "transformers>=4.49,<5" "accelerate>=1.4,<2" "huggingface-hub>=0.29,<1" "datasets>=3.2,<5" "sentencepiece>=0.2" "safetensors>=0.5" "scikit-learn>=1.4" "joblib>=1.3" "matplotlib>=3.7" "pandas>=2.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 1 — Configure one training session

Pick one task and 2–3 models per session. The notebook stores model checkpoints under a task/model directory. **Do not turn FINALIZE on until your chosen candidate set is complete**. For a repeat-seed replication, use another independent `SEED` and a new output directory.

Recommended:
1. `sentiment_raw`: baseline + HingRoBERTa-Mixed; run `sentiment` as a separate processed-source ablation (not identical v4 cleaning).
2. Cyberbullying7 baseline + HingRoBERTa-Mixed.
3. Cyberbullying binary (derived labels) baseline + Twitter-XLM-R; then use MuRIL/XLM-R as compute allows.

*Colab uses Google Drive for durable checkpoints. Kaggle `/kaggle/working` does not persist between fresh sessions unless you commit/version outputs or restore them from a Kaggle dataset.*

In [2]:
import sys,os,json,time,platform
from pathlib import Path
import pandas as pd, numpy as np, torch
from IPython.display import display

SEED=42
RUN_TASKS=['cyberbullying7','sentiment_raw']  # Existing Drive tasks; remove tasks you do not need.
RUN_MODELS=['tfidf_word_char','tfidf_svm','hing_roberta_mixed']
PREPARE_MISSING_TASKS=False  # Never redownload/rebuild splits automatically.
RUN_MISSING_MODELS=False  # Never start training unless explicitly enabled.
# Other valid names: tfidf_word, tfidf_char, tfidf_normalized,
# bilstm_last, bilstm_meanmax, bilstm_attention, muril, hingbert,
# xlmr, twitter_xlmr, twitter_xlmr_sentiment, distilbert
USE_COLAB_DRIVE=('google.colab' in sys.modules)
EPOCHS_BILSTM=7
EPOCHS_TRANSFORMER=3
MAX_LENGTH=128
BUILD_BLEND=False  # enable after >=2 candidate fits, BEFORE final lock
FINALIZE=False     # only once after the prespecified candidate list is complete
EXTERNAL_CHALLENGE_CSV=''  # optional independently human-annotated CSV (text,label)
HASOC_TRAIN_CSV=''         # authorized HASOC 2021 code-mix files only
HASOC_TEST_CSV=''
MC_LABEL_MAP=None          # supply only if upstream uses numeric IDs; use verified source codebook
MC_TEXT_COLUMN=None
MC_LABEL_COLUMN=None

in_colab='google.colab' in sys.modules
in_kaggle=Path('/kaggle/working').exists()
if USE_COLAB_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
base=(Path('/content/drive/MyDrive') if USE_COLAB_DRIVE else
      Path('/kaggle/working') if in_kaggle else
      Path('/content') if in_colab else Path.cwd())
ROOT=base/f'MixBhasha_V5_seed{SEED}'
ROOT.mkdir(parents=True,exist_ok=True)
print('ROOT',ROOT,'torch',torch.__version__,'GPU',torch.cuda.is_available())
if not torch.cuda.is_available():print('GPU absent: classical models and small LSTM are still usable.')

Mounted at /content/drive
ROOT /content/drive/MyDrive/MixBhasha_V5_seed42 torch 2.14.1+cpu GPU False
GPU absent: classical models and small LSTM are still usable.


## 2 — Self-contained training engine

This cell refreshes the project source from `main` and loads `src/mixbhasha_core.py`, so notebook runs use the same checkpoint and data-safety fixes as the repository.

In [3]:
CORE_CODE='"""MixBhasha v5: benchmark-aware code-mixed classification experiments."""\nfrom __future__ import annotations\nimport hashlib, html, json, os, random, re, unicodedata, warnings, gc, inspect\nfrom collections import Counter\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import FeatureUnion, Pipeline\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import LinearSVC\nfrom sklearn.calibration import CalibratedClassifierCV\nfrom sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,\n    balanced_accuracy_score, average_precision_score, confusion_matrix,\n    classification_report, brier_score_loss)\n\nSEED=42\nLABELS={\'sentiment\':[\'negative\',\'neutral\',\'positive\'],\n        \'sentiment_raw\':[\'negative\',\'neutral\',\'positive\'],\n        \'cyberbullying7\':[\'not_cyberbullying\',\'age\',\'gender\',\'religion\',\'mockery\',\'abusive\',\'offensive\'],\n        \'cyberbullying_binary\':[\'not_cyberbullying\',\'cyberbullying_category\'],\n        \'offense_hasoc\':[\'not_hate_offensive\',\'hate_or_offensive\'],\n        \'hate\':[\'non_hate\',\'hate\']}\nSENT_URLS={\n \'train\':\'https://raw.githubusercontent.com/singhnivedita/SemEval2020-Task9/master/Fully%20Processed%20Datasets/FinalTrainingOnly.tsv\',\n \'test\':\'https://raw.githubusercontent.com/singhnivedita/SemEval2020-Task9/master/Fully%20Processed%20Datasets/ValidationOnly.tsv\'}\nRAW_SENT_URLS={\n \'train\':\'https://raw.githubusercontent.com/keshav22bansal/BAKSA_IITK/master/data/hinglish/train.txt\',\n \'test\':\'https://raw.githubusercontent.com/keshav22bansal/BAKSA_IITK/master/data/hinglish/test.txt\'}\nMC_REPO=\'sahinurlaskar/MC-Hinglish1.0\'\nMC_FILES={\'train\':\'Train_data_MC-Hinglish1.0.csv\',\'test\':\'Test_data_MC-Hinglish1.0.csv\'}\nPRISM_REPO=\'pankajbiswas6/prism-hinglish-hate-speech\'\nTRANSFORMERS={\n \'distilbert\':\'distilbert/distilbert-base-uncased\',\n \'muril\':\'google/muril-base-cased\',\n \'hingbert\':\'l3cube-pune/hing-bert\',\n \'xlmr\':\'FacebookAI/xlm-roberta-base\',\n \'hing_roberta_mixed\':\'l3cube-pune/hing-roberta-mixed\',\n \'twitter_xlmr\':\'cardiffnlp/twitter-xlm-roberta-base\',\n \'twitter_xlmr_sentiment\':\'cardiffnlp/twitter-xlm-roberta-base-sentiment\'}\nMODEL_NAMES=[\'tfidf_word\',\'tfidf_char\',\'tfidf_word_char\',\'tfidf_normalized\',\'tfidf_svm\',\n             \'bilstm_last\',\'bilstm_meanmax\',\'bilstm_attention\',*TRANSFORMERS.keys()]\nURL_RX=re.compile(r\'(?:https?://|www\\.)\\S+\',re.I)\nMENTION_RX=re.compile(r\'(?<!\\w)@[\\w]+\',re.UNICODE)\nSPACE_RX=re.compile(r\'\\s+\')\nTOKEN_RX=re.compile(r\'(?u)\\w+|[^\\w\\s]\')\nALIASES={\'bohot\':\'bahut\',\'bohut\':\'bahut\',\'bhot\':\'bahut\',\'bht\':\'bahut\',\n         \'achha\':\'acha\',\'accha\':\'acha\',\'achaa\':\'acha\',\'nhi\':\'nahi\',\'nahin\':\'nahi\',\n         \'kyaaa\':\'kya\',\'plzz\':\'please\'}\n\ndef set_seed(seed=SEED):\n    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n\ndef clean(text):\n    text=unicodedata.normalize(\'NFC\',html.unescape(str(text)))\n    text=URL_RX.sub(\' HTTPURL \',text)\n    text=MENTION_RX.sub(\' @USER \',text)\n    return SPACE_RX.sub(\' \',text).strip()\n\ndef normalize_roman(text):\n    return re.sub(r\'(?u)\\b\\w+\\b\',lambda m:ALIASES.get(m.group(0).lower(),m.group(0)),str(text)).lower()\n\ndef output_paths(root):\n    root=Path(root)\n    for name in [\'data\',\'models\',\'results\',\'private_analysis\']:(root/name).mkdir(parents=True,exist_ok=True)\n    return root\n\ndef download_once(url,dest):\n    from urllib.request import urlopen,Request\n    dest=Path(dest);dest.parent.mkdir(parents=True,exist_ok=True)\n    if not dest.exists() or dest.stat().st_size<100:\n        try:\n            with urlopen(Request(url,headers={\'User-Agent\':\'MixBhasha-v3-academic\'}),timeout=120) as resp:\n                b=resp.read()\n            if len(b)<100:raise ValueError(\'file unexpectedly short\')\n            tmp=dest.with_name(dest.name+\'.partial\')\n            tmp.write_bytes(b)\n            tmp.replace(dest)\n        except Exception as ex:raise RuntimeError(f\'Cannot fetch {url}. Enable internet or supply a local copy.\') from ex\n    return dest\n\ndef _sent_tsv(path):\n    df=pd.read_csv(path,sep=\'\\t\',header=None,names=[\'uid\',\'text\',\'label\'],\n                   dtype=str,on_bad_lines=\'warn\')\n    df=df[df.label.isin([\'0\',\'1\',\'2\'])].copy()\n    df[\'label\']=df.label.astype(int)\n    return df[[\'text\',\'label\']]\n\ndef _sent_raw_tsv(path):\n    """Team-released SentiMix raw Hinglish text with header: uid, text, label.\n\n    This is a public team copy, NOT a claim of official organizer gold test release.\n    """\n    df=pd.read_csv(path,sep=\'\\t\',dtype=str,on_bad_lines=\'error\')\n    cols={str(x).strip().lower():x for x in df.columns}\n    if not {\'text\',\'label\'}.issubset(cols):\n        raise ValueError(f\'Raw Hinglish mirror changed schema; required text,label: {list(df.columns)}\')\n    out=df[[cols[\'text\'],cols[\'label\']]].copy()\n    out.columns=[\'text\',\'label\']\n    bad=set(out.label.dropna().astype(str)) - {\'0\',\'1\',\'2\'}\n    if bad:raise ValueError(f\'Unexpected raw sentiment labels {sorted(bad)}; verify label codebook\')\n    out=out.dropna(subset=[\'text\',\'label\'])\n    out[\'label\']=out.label.astype(int)\n    return out\n\n\ndef _col(df,options):\n    canon={re.sub(r\'[^a-z0-9]\',\'\',str(c).lower()):c for c in df.columns}\n    for item in options:\n        v=canon.get(re.sub(r\'[^a-z0-9]\',\'\',item.lower()))\n        if v is not None:return v\n    raise ValueError(f\'Cannot identify one of {options}. CSV columns: {list(df.columns)}. \'\n                     \'Set CSV column headers to text,label manually; no guessed labels are allowed.\')\n\ndef _canonical_label(value):\n    """Standardize explicit strings, never infer what an integer class ID means."""\n    return re.sub(r\'[^a-z0-9]+\',\'\',str(value).casefold())\n\nMC_KNOWN_CATEGORIES={\n    \'notcyberbullying\',\'notbullying\',\'noncyberbullying\',\n    \'offensive\',\'abusive\',\'age\',\'gender\',\'religion\',\'mockery\'\n}\n\ndef _offense_mc(df, label_map=None, text_column=None, label_column=None):\n    """Restricted MC-Hinglish subset; numeric IDs require a dataset-documented map.\n\n    `label_map` should map source IDs/names to actual original category strings,\n    NOT directly to 0/1. E.g., {\'0\':\'Not-Cyberbullying\',\'4\':\'Offensive\'} only\n    when verified from the particular source release. Unknown categories excluded.\n    """\n    tcol=text_column if text_column is not None else _col(df, [\n        \'text\',\'tweet\',\'tweets\',\'tweet text\',\'comment\',\'comments\',\'sentence\',\n        \'post\',\'post text\',\'content\',\'message\',\'hinglish text\'])\n    if tcol not in df.columns:\n        raise ValueError(f\'MC text_column {tcol!r} not in CSV columns {list(df.columns)}\')\n    opts=[\'label\',\'labels\',\'class\',\'class id\',\'class_id\',\'category\',\'type\',\'tag\',\'cyberbullying\',\'target\']\n    candidates=[label_column] if label_column else [\n        df.columns[i] for i,c in enumerate(df.columns)\n        if _canonical_label(c) in {_canonical_label(x) for x in opts}]\n    if not candidates:raise ValueError(f\'No MC label column found. Columns={list(df.columns)}\')\n    positive={\'offensive\',\'abusive\'}\n    negative={\'notcyberbullying\',\'notbullying\',\'noncyberbullying\'}\n    normalized_map={_canonical_label(k):_canonical_label(v) for k,v in (label_map or {}).items()}\n    invalid=set(normalized_map.values())-MC_KNOWN_CATEGORIES\n    if invalid:raise ValueError(f\'Unknown mapped MC categories {sorted(invalid)}; use documented category names\')\n    scored=[]\n    for c in candidates:\n        if c not in df.columns:raise ValueError(f\'MC label_column {c!r} absent. Columns={list(df.columns)}\')\n        raw=df[c].fillna(\'\').map(_canonical_label)\n        numeric=(raw.str.fullmatch(r\'\\d+\').fillna(False)).any()\n        if numeric and not normalized_map:\n            raise ValueError(f\'MC column {c!r} contains numeric class IDs. \'\n                             \'Provide MC_LABEL_MAP based on the dataset codebook; do not guess IDs.\')\n        if normalized_map:\n            unknown_ids=sorted(set(raw[raw.str.fullmatch(r\'\\d+\').fillna(False)])-set(normalized_map))\n            if unknown_ids:raise ValueError(f\'Unmapped numeric MC IDs in {c!r}: {unknown_ids}. \'\n                                               \'Provide a complete verified MC_LABEL_MAP.\')\n        mapped=raw.map(lambda x:normalized_map.get(x,x))\n        n=int(mapped.isin(positive|negative).sum())\n        scored.append((n,c,mapped))\n    scored.sort(key=lambda z:z[0],reverse=True)\n    hits,lcol,lbl=scored[0]\n    if hits==0:raise ValueError(f\'No recognized MC categories. Values={df[lcol].value_counts(dropna=False).head(12).to_dict()}\')\n    if len(scored)>1 and scored[1][0]==hits and label_column is None:\n        raise ValueError(f\'Ambiguous label columns {lcol!r}, {scored[1][1]!r}; set MC_LABEL_COLUMN explicitly\')\n    chosen=lbl.isin(positive|negative)\n    out=pd.DataFrame({\'text\':df.loc[chosen,tcol].values,\n        \'label\':lbl.loc[chosen].isin(positive).astype(int).values})\n    class_counts=lbl.value_counts(dropna=False).to_dict()\n    out.attrs[\'source_mapping\']={\n        \'text_column\':str(tcol),\'label_column\':str(lcol),\n        \'raw_rows\':int(len(df)),\'included_rows\':int(len(out)),\n        \'excluded_rows\':int((~chosen).sum()),\n        \'source_categories\':{str(k):int(v) for k,v in class_counts.items()},\n        \'positive_source_categories\':[\'Offensive\',\'Abusive\'],\n        \'negative_source_categories\':[\'Not-Cyberbullying\'],\n        \'scope\':\'restricted binary target, not all offensive/hateful/cyberbullying content\'}\n    return out\n\ndef _offense_hasoc(df):\n    tcol=_col(df,[\'text\',\'tweet\',\'tweets\',\'comment\',\'post\',\'content\'])\n    lcol=_col(df,[\'label\',\'class\',\'category\',\'task1\',\'task_1\',\'tag\'])\n    lbl=df[lcol].fillna(\'\').astype(str).str.upper().str.strip()\n    allowed={\'HOF\':1,\'NOT\':0}\n    bad=sorted(set(lbl)-set(allowed))\n    if bad:raise ValueError(f\'Unexpected HASOC labels {bad}; expected exact HOF,NOT. \'\n                             \'Do not silently convert HATE to OFFENSIVE.\')\n    return pd.DataFrame({\'text\':df[tcol], \'label\':lbl.map(allowed).astype(int)})\n\ndef _normalize(df,task):\n    df=df.copy().dropna(subset=[\'text\',\'label\'])\n    df[\'text\']=df.text.astype(str).map(clean)\n    df=df[df.text.str.len().between(2,4000)].copy()\n    df[\'label\']=pd.to_numeric(df.label,errors=\'raise\').astype(int)\n    if not set(df.label.unique()).issubset(set(range(len(LABELS[task])))):\n        raise ValueError(f\'Invalid label in {task}: {df.label.unique().tolist()}\')\n    return df.reset_index(drop=True)\n\ndef _dedupe(frames,task,require_all_splits=True):\n    """Quarantine contradictions globally; protect holdout before validation/train.\n\n    All records with conflicting labels for the same normalized text are removed,\n    not assigned to the arbitrary first-seen class. No raw user text is logged.\n    Holdout precedence: test > val > train for identically labelled duplicates.\n    """\n    order=[split for split in [\'test\',\'val\',\'train\'] if split in frames]\n    normalized={}; report={}\n    for split in order:\n        original=frames[split]\n        df=_normalize(original,task)\n        df[\'_key\']=df.text.str.casefold().str.strip()\n        normalized[split]=df\n        report[split]={\n            \'input_rows\':int(len(original)),\n            \'post_normalization_rows\':int(len(df)),\n            \'invalid_or_empty_rows_removed\':int(len(original)-len(df)),\n            \'within_split_conflicting_text_keys\':int(df.groupby(\'_key\').label.nunique().gt(1).sum())}\n    all_pairs=pd.concat([normalized[k][[\'_key\',\'label\']] for k in order],ignore_index=True)\n    contradictory=set(all_pairs.groupby(\'_key\').label.nunique().loc[lambda z:z>1].index)\n    # Hash-only examples allow human audit without exporting raw text.\n    digest=[hashlib.sha256(k.encode(\'utf-8\')).hexdigest()[:16] for k in sorted(contradictory)]\n    global_report={\'contradictory_text_keys\':len(contradictory),\n                   \'contradictory_rows_removed\':int(all_pairs[\'_key\'].isin(contradictory).sum()),\n                   \'conflicting_key_hash_samples\':digest[:20],\n                   \'policy\':\'quarantine every occurrence of a text with conflicting labels\'}\n    seen=set();out={}\n    for split in order:\n        df=normalized[split]\n        conflicts=df[\'_key\'].isin(contradictory)\n        good=df.loc[~conflicts].copy()\n        same=good[\'_key\'].duplicated(keep=\'first\')\n        good=good.loc[~same].copy()\n        across=good[\'_key\'].isin(seen)\n        keep=good.loc[~across].copy()\n        seen.update(keep[\'_key\'].tolist())\n        kept=keep.drop(columns=\'_key\').reset_index(drop=True)\n        r=report[split]\n        r.update({\'conflicting_rows_removed\':int(conflicts.sum()),\n                  \'same_label_duplicates_removed\':int(same.sum()),\n                  \'cross_split_overlap_rows_removed\':int(across.sum()),\n                  \'kept\':int(len(kept)),\n                  \'removed_total\':int(r[\'input_rows\']-len(kept)),\n                  \'class_counts\':{str(k):int(v) for k,v in kept.label.value_counts().sort_index().items()}})\n        if require_all_splits and (len(kept)<15 or kept.label.nunique()!=len(LABELS[task])):\n            raise ValueError(f\'{task}/{split}: too small or missing classes after audited dedup: {r}\')\n        if len(kept)==0:raise ValueError(f\'{task}/{split}: no usable data left after dedup\')\n        out[split]=kept\n    report[\'_global\']=global_report\n    return out,report\n\ndef _mc_all(df, task, label_map=None, text_column=None, label_column=None):\n    """Use *all seven* original categories. No undocumented numeric label guesses."""\n    if task not in (\'cyberbullying7\',\'cyberbullying_binary\'):\n        raise ValueError(\'MC full-label loader requires cyberbullying task\')\n    tcol=text_column or _col(df,[\'tweets\',\'tweet\',\'text\',\'comment\',\'post\',\'content\',\'message\'])\n    lcol=label_column or _col(df,[\'cyberbullying_type\',\'category\',\'label\',\'class\',\'type\'])\n    raw=df[lcol].fillna(\'\').map(_canonical_label)\n    normalized_map={_canonical_label(k):_canonical_label(v) for k,v in (label_map or {}).items()}\n    if raw.str.fullmatch(r\'\\d+\').fillna(False).any() and not normalized_map:\n        raise ValueError(\'Numeric MC labels need an explicit *source-category* mapping; refusing to guess\')\n    unknown_ids=set(raw[raw.str.fullmatch(r\'\\d+\').fillna(False)])-set(normalized_map)\n    if unknown_ids:raise ValueError(f\'Unmapped MC numeric IDs: {sorted(unknown_ids)}\')\n    cat=raw.map(lambda x:normalized_map.get(x,x))\n    canon=list(LABELS[\'cyberbullying7\'])\n    canon_lookup={_canonical_label(name):i for i,name in enumerate(canon)}\n    unexpected=sorted(set(cat)-set(canon_lookup))\n    if unexpected:raise ValueError(f\'Unexpected MC category: {unexpected[:20]}. Update only with verified source taxonomy.\')\n    labels=cat.map(canon_lookup).astype(int)\n    if task==\'cyberbullying_binary\':labels=(labels!=0).astype(int)\n    out=pd.DataFrame({\'text\':df[tcol].values,\'label\':labels.values})\n    out.attrs[\'source_mapping\']={\'raw_rows\':int(len(df)), \'included_rows\':int(len(out)),\n        \'categories\':{str(k):int(v) for k,v in cat.value_counts().items()},\n        \'task_label_scope\':(\'seven mutually exclusive annotated MC cyberbullying categories\' if task==\'cyberbullying7\' else\n                          \'derived binary: any of the six harmful categories vs not-cyberbullying\'),\n        \'note\':\'Harmful-category is NOT equivalent to all offensive language or hate speech\'}\n    return out\n\n\ndef _key_series(df):\n    return df.text.astype(str).str.casefold().str.strip()\n\n\ndef _dedupe_train_only(pool,task,excluded_texts=()):\n    """Train-label-only cleaning. Holdout labels are NEVER consulted to curate train."""\n    x=_normalize(pool,task)\n    x[\'_key\']=_key_series(x)\n    contradictory=set(x.groupby(\'_key\').label.nunique().loc[lambda n:n>1].index)\n    mask_conflict=x._key.isin(contradictory)\n    x1=x.loc[~mask_conflict].copy()\n    mask_repeat=x1._key.duplicated()\n    x2=x1.loc[~mask_repeat].copy()\n    forbid=set(excluded_texts)\n    mask_overlap=x2._key.isin(forbid)\n    clean=x2.loc[~mask_overlap].drop(columns=[\'_key\']).reset_index(drop=True)\n    audit={\'input_rows\':len(pool),\'invalid_rows_removed\':int(len(pool)-len(x)),\n           \'conflicting_keys_quarantined\':len(contradictory),\'conflicting_rows_removed\':int(mask_conflict.sum()),\n           \'duplicate_same_label_removed\':int(mask_repeat.sum()),\n           \'overlap_with_reserved_text_removed\':int(mask_overlap.sum()),\n           \'remaining_rows\':int(len(clean)),\n           \'conflicting_key_sha256_prefixes\':[hashlib.sha256(k.encode(\'utf8\')).hexdigest()[:16] for k in sorted(contradictory)[:20]]}\n    return clean,audit\n\n\ndef strict_split_source(pool, holdout, task, seed=SEED, validation_fraction=.12):\n    """Test labels read for reporting only, never used to drop/retain training data."""\n    reserved=_normalize(holdout,task)\n    # IMPORTANT: no operations involving reserved[\'label\'] decide train membership.\n    reserved_keys=set(_key_series(reserved))\n    cleaned,audit=_dedupe_train_only(pool,task,reserved_keys)\n    count=cleaned.label.value_counts().reindex(range(len(LABELS[task])),fill_value=0)\n    if (count<3).any():raise ValueError(f\'{task} has insufficient examples after cleaning: {count.to_dict()}\')\n    tr,va=train_test_split(cleaned,test_size=validation_fraction,stratify=cleaned.label,random_state=seed)\n    out={\'train\':tr.reset_index(drop=True),\'val\':va.reset_index(drop=True),\'test\':reserved.reset_index(drop=True)}\n    tests=_key_series(reserved)\n    audit[\'test_policy\']={\'rows_before\':len(holdout),\'rows_after_text_filter\':len(reserved),\n                          \'same_text_duplicate_rows\':int(tests.duplicated().sum()),\n                          \'source_holdout_labels_used_to_filter_training\':False,\n                          \'holdout_label_cleanup\':\'No label-dependent row removals\',\n                          \'previously_exposed_v4_test_for_sentiment\':task==\'sentiment\',\n                          \'sentiMix_related_potential_overlap\':task in (\'sentiment\',\'sentiment_raw\')}\n    groups=[set(_key_series(out[k])) for k in (\'train\',\'val\',\'test\')]\n    assert not (groups[0]&groups[1] or groups[0]&groups[2] or groups[1]&groups[2])\n    return out,audit\n\n\ndef strict_presplit_source(original,task):\n    """Use existing PRISM splits, clean train labels only; remove text-key overlaps without test labels."""\n    reserved=_normalize(original[\'test\'],task)\n    val=_normalize(original[\'val\'],task)\n    testkeys=set(_key_series(reserved))\n    val=val.loc[~_key_series(val).isin(testkeys)].reset_index(drop=True)\n    forbidden=testkeys|set(_key_series(val))\n    tr,audit=_dedupe_train_only(original[\'train\'],task,forbidden)\n    out={\'train\':tr,\'val\':val,\'test\':reserved}\n    return out,{\'train_cleaning\':audit, \'policy\':\'preserve source partitions; no test-label curation\'}\n\n\ndef load_task(task,root,offense_source=\'mc\',hasoc_train=\'\',hasoc_test=\'\',\n              mc_label_map=None,mc_text_column=None,mc_label_column=None):\n    root=output_paths(root)\n    if task==\'sentiment\':\n        pool=_sent_tsv(download_once(SENT_URLS[\'train\'],root/\'data\'/\'sentiment_train_source.tsv\'))\n        test=_sent_tsv(download_once(SENT_URLS[\'test\'],root/\'data\'/\'sentiment_dev_source.tsv\'))\n        source=\'Third-party processed SentiMix: published dev previously exposed by v4, not official test\'\n    elif task==\'sentiment_raw\':\n        pool=_sent_raw_tsv(download_once(RAW_SENT_URLS[\'train\'],root/\'data\'/\'raw_sentimix_train_team_mirror.tsv\'))\n        test=_sent_raw_tsv(download_once(RAW_SENT_URLS[\'test\'],root/\'data\'/\'raw_sentimix_test_team_mirror.tsv\'))\n        source=\'BAKSA public team mirror of SentiMix Hinglish raw text; NOT organizer-verified official leaderboard test\'\n    elif task in (\'cyberbullying7\',\'cyberbullying_binary\'):\n        from huggingface_hub import hf_hub_download\n        paths={k:hf_hub_download(repo_id=MC_REPO,filename=v,repo_type=\'dataset\',\n                local_dir=str(root/\'data\'/\'mc_source\')) for k,v in MC_FILES.items()}\n        mapped={k:_mc_all(pd.read_csv(path),task,mc_label_map,mc_text_column,mc_label_column)\n                for k,path in paths.items()}\n        pool,test=mapped[\'train\'],mapped[\'test\']\n        source=\'MC-Hinglish1.0 all 7 categories; derived binary supported but separately named\'\n    elif task==\'offense_hasoc\':\n        if not (hasoc_train and hasoc_test and Path(hasoc_train).is_file() and Path(hasoc_test).is_file()):\n            raise FileNotFoundError(\'Authorized HASOC 2021 code-mix CSVs must be supplied with HOF/NOT labels\')\n        pool=_offense_hasoc(pd.read_csv(hasoc_train))\n        test=_offense_hasoc(pd.read_csv(hasoc_test))\n        source=\'HASOC 2021 authorized Subtask 2; context-sensitive HOF/NOT\'\n    elif task==\'hate\':\n        from datasets import load_dataset\n        ds=load_dataset(PRISM_REPO)\n        if not {\'train\',\'val\',\'test\'}.issubset(set(ds)):\n            raise ValueError(\'PRISM requires original train/val/test splits\')\n        original={k:ds[k].to_pandas()[[\'text\',\'label\',\'lang\']] for k in (\'train\',\'val\',\'test\')}\n        # this small optional task retains only text,label; language analysis can run directly on original.\n        frames,audit=strict_presplit_source(original,task)\n        return frames,\'PRISM hate != generic offensive-language moderation\',audit\n    else:raise ValueError(f\'Unsupported task {task}\')\n    frames,audit=strict_split_source(pool,test,task)\n    if task in (\'cyberbullying7\',\'cyberbullying_binary\'):\n        audit[\'source_categories\']=mapped[\'train\'].attrs[\'source_mapping\']\n    return frames,source,audit\n\ndef prepare(root,tasks=(\'sentiment\',\'cyberbullying7\'),offense_source=\'mc\',hasoc_train=\'\',hasoc_test=\'\',\n            mc_label_map=None,mc_text_column=None,mc_label_column=None):\n    root=output_paths(root);all_data={};report={}\n    for task in tasks:\n        frames,source,audit=load_task(task,root,offense_source,hasoc_train,hasoc_test,\n                                     mc_label_map,mc_text_column,mc_label_column)\n        # Do not overwrite frozen splits if a previous experiment used a different source.\n        existing={split:root/\'data\'/f\'{task}_{split}.csv\' for split in [\'train\',\'val\',\'test\']}\n        if all(p.exists() for p in existing.values()):\n            prev=load_prepared(root,[task])[task]\n            if fingerprint(prev)!=fingerprint(frames):\n                if any((root/\'models\'/task).rglob(\'validated.json\')):\n                    raise RuntimeError(f\'{task}: prepared dataset differs from a trained experiment. \'\n                        \'Start with a NEW ROOT; refusing to mix old models and new splits.\')\n        for split,d in frames.items():d.to_csv(existing[split],index=False)\n        all_data[task]=frames\n        report[task]={\'source\':source,\'split_sha256\':fingerprint(frames),\'splits\':audit}\n    audit_file=root/\'results\'/\'data_audit.json\'\n    previous=json.loads(audit_file.read_text(encoding=\'utf8\')) if audit_file.exists() else {}\n    previous.update(report)\n    audit_file.write_text(json.dumps(previous,indent=2,ensure_ascii=False),encoding=\'utf8\')\n    return all_data,report\n\ndef load_prepared(root,tasks):\n    root=Path(root)\n    data={}\n    for task in tasks:\n        data[task]={s:pd.read_csv(root/\'data\'/f\'{task}_{s}.csv\').fillna({\'text\':\'\'}) for s in [\'train\',\'val\',\'test\']}\n        for s in data[task]:data[task][s][\'label\']=data[task][s][\'label\'].astype(int)\n    return data\n\ndef fingerprint(frames):\n    h=hashlib.sha256()\n    for name in [\'train\',\'val\',\'test\']:\n        d=frames[name]\n        h.update(name.encode())\n        for row in d[[\'text\',\'label\']].itertuples(index=False,name=None):\n            h.update((str(row[0])+\'\\t\'+str(row[1])+\'\\n\').encode(\'utf-8\'))\n    return h.hexdigest()\n\ndef feature_model(name):\n    word=lambda pre=None:TfidfVectorizer(analyzer=\'word\',ngram_range=(1,2),min_df=2,\n                                      max_features=45000,sublinear_tf=True,preprocessor=pre)\n    char=lambda pre=None:TfidfVectorizer(analyzer=\'char\',ngram_range=(3,5),min_df=2,\n                                      max_features=50000,sublinear_tf=True,preprocessor=pre)\n    if name==\'tfidf_svm\':\n        v=FeatureUnion([(\'word\',word()),(\'char\',char())])\n        classifier=CalibratedClassifierCV(estimator=LinearSVC(C=1.0,class_weight=\'balanced\'),method=\'sigmoid\',cv=3)\n        return Pipeline([(\'vectorizer\',v),(\'classifier\',classifier)])\n    if name==\'tfidf_word\':v=word()\n    elif name==\'tfidf_char\':v=char()\n    elif name==\'tfidf_word_char\':v=FeatureUnion([(\'word\',word()),(\'char\',char())])\n    elif name==\'tfidf_normalized\':v=FeatureUnion([(\'word\',word(normalize_roman)),(\'char\',char(normalize_roman))])\n    else:raise ValueError(name)\n    return Pipeline([(\'vectorizer\',v),(\'classifier\',LogisticRegression(\n        max_iter=1500,class_weight=\'balanced\',solver=\'lbfgs\'))])\n\ndef tokens(text):return TOKEN_RX.findall(str(text).lower())\ndef vocab_build(texts,max_vocab=35000,min_freq=2):\n    c=Counter(t for x in texts for t in tokens(x))\n    d={\'<PAD>\':0,\'<UNK>\':1}\n    for k,n in c.most_common():\n        if len(d)>=max_vocab:break\n        if n>=min_freq:d[k]=len(d)\n    return d\n\ndef encode(text,vocab,max_len=110):return [vocab.get(t,1) for t in tokens(text)[:max_len]] or [1]\n\nclass Words(Dataset):\n    def __init__(self,df,vocab):\n        self.x=[encode(t,vocab) for t in df.text];self.y=list(map(int,df.label))\n    def __len__(self):return len(self.x)\n    def __getitem__(self,i):return self.x[i],self.y[i]\n\ndef batch_words(rows):\n    ids,y=zip(*rows)\n    lens=torch.tensor([len(s) for s in ids])\n    x=nn.utils.rnn.pad_sequence([torch.tensor(s,dtype=torch.long) for s in ids],batch_first=True,padding_value=0)\n    return x,lens,torch.tensor(y,dtype=torch.long)\n\nclass BiLSTM(nn.Module):\n    def __init__(self,n_vocab,n_labels,pool=\'attention\',emb=128,hidden=96):\n        super().__init__()\n        if pool not in [\'last\',\'meanmax\',\'attention\']:raise ValueError(pool)\n        self.pool=pool\n        self.emb=nn.Embedding(n_vocab,emb,padding_idx=0)\n        self.rnn=nn.LSTM(emb,hidden,batch_first=True,bidirectional=True)\n        if pool==\'attention\':self.att=nn.Linear(hidden*2,1)\n        dims=hidden*4 if pool==\'meanmax\' else hidden*2\n        self.head=nn.Sequential(nn.Dropout(.35),nn.Linear(dims,64),nn.ReLU(),nn.Dropout(.2),nn.Linear(64,n_labels))\n    def forward(self,ids,lens):\n        x=self.emb(ids)\n        packed=nn.utils.rnn.pack_padded_sequence(x,lens.cpu(),batch_first=True,enforce_sorted=False)\n        out,(h,_) = self.rnn(packed)\n        if self.pool==\'last\':feat=torch.cat([h[-2],h[-1]],dim=-1)\n        else:\n            out,_=nn.utils.rnn.pad_packed_sequence(out,batch_first=True,total_length=ids.size(1))\n            mask=torch.arange(ids.size(1),device=ids.device)[None,:]<lens.to(ids.device)[:,None]\n            if self.pool==\'attention\':\n                scores=self.att(out).squeeze(-1).masked_fill(~mask,-1e4)\n                weights=torch.softmax(scores,dim=1)\n                feat=(out*weights.unsqueeze(-1)).sum(1)\n            else:\n                mean=(out*mask.unsqueeze(-1)).sum(1)/lens.to(ids.device).clamp_min(1).unsqueeze(1)\n                mx=out.masked_fill(~mask.unsqueeze(-1),-1e4).max(dim=1).values\n                feat=torch.cat([mean,mx],dim=-1)\n        return self.head(feat)\n\n@torch.no_grad()\ndef neural_predict(model,df,vocab,device,batch_size=128):\n    loader=DataLoader(Words(df,vocab),batch_size=batch_size,collate_fn=batch_words,shuffle=False)\n    model.eval();parts=[]\n    for x,l,_ in loader:\n        y=model(x.to(device),l)\n        parts.append(torch.softmax(y,dim=-1).cpu().numpy())\n    return np.concatenate(parts,axis=0)\n\ndef fit_bilstm(train,val,task,model_name,path,device,epochs=8,batch_size=64,seed=SEED):\n    path=Path(path);path.mkdir(parents=True,exist_ok=True)\n    pool=model_name[len(\'bilstm_\'):]\n    vocab_file=path/\'vocab.json\';checkpoint=path/\'train_state.pt\';best_file=path/\'best.pt\'\n    if vocab_file.exists():vocab=json.loads(vocab_file.read_text(encoding=\'utf8\'))\n    else:\n        vocab=vocab_build(train.text)\n        vocab_file.write_text(json.dumps(vocab,ensure_ascii=False),encoding=\'utf8\')\n    net=BiLSTM(len(vocab),len(LABELS[task]),pool=pool).to(device)\n    count=train.label.value_counts().reindex(range(len(LABELS[task])),fill_value=1).to_numpy()\n    weights=np.clip(len(train)/(len(LABELS[task])*count),.5,4.)\n    loss_fn=nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=device))\n    opt=torch.optim.AdamW(net.parameters(),lr=.001,weight_decay=.01)\n    start,best,bad=0,-1,0\n    if checkpoint.exists():\n        state=torch.load(checkpoint,map_location=device,weights_only=False)\n        net.load_state_dict(state[\'net\']);opt.load_state_dict(state[\'opt\'])\n        start,best,bad=state[\'epoch\'],state[\'best\'],state[\'bad\']\n        if \'torch_rng\' in state:torch.set_rng_state(state[\'torch_rng\'].cpu())\n        if torch.cuda.is_available() and \'cuda_rng\' in state:torch.cuda.set_rng_state_all(state[\'cuda_rng\'])\n        print(f\'Resuming {task}/{model_name} at epoch {start+1}\')\n    loader=DataLoader(Words(train,vocab),batch_size=batch_size,shuffle=True,collate_fn=batch_words)\n    for epoch in range(start+1,epochs+1):\n        net.train();total=0\n        for x,l,y in loader:\n            opt.zero_grad(set_to_none=True)\n            logits=net(x.to(device),l);loss=loss_fn(logits,y.to(device))\n            loss.backward();nn.utils.clip_grad_norm_(net.parameters(),1.);opt.step();total+=loss.item()*len(y)\n        prob=neural_predict(net,val,vocab,device)\n        f1=f1_score(val.label,prob.argmax(1),average=\'macro\',zero_division=0)\n        if f1>best+1e-5:\n            best=f1;bad=0;torch.save(net.state_dict(),best_file)\n        else:bad+=1\n        state={\'net\':net.state_dict(),\'opt\':opt.state_dict(),\'epoch\':epoch,\'best\':best,\'bad\':bad,\n               \'torch_rng\':torch.get_rng_state()}\n        if torch.cuda.is_available():state[\'cuda_rng\']=torch.cuda.get_rng_state_all()\n        torch.save(state,checkpoint)\n        print(f\'{task}/{model_name}: epoch={epoch} loss={total/len(train):.4f} val_argmax_f1={f1:.4f}\')\n        if bad>=2:break\n    if not best_file.exists():raise RuntimeError(\'BiLSTM best checkpoint missing\')\n    net.load_state_dict(torch.load(best_file,map_location=device,weights_only=True))\n    return net,vocab\n\ndef _hf_imports():\n    try:\n        from transformers import AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments,DataCollatorWithPadding\n        from transformers.trainer_utils import get_last_checkpoint\n        return AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments,DataCollatorWithPadding,get_last_checkpoint\n    except ImportError as e:raise ImportError(\'Install requirements-train.txt to run transformer training\') from e\n\nclass TokenData(Dataset):\n    """Tokenize by individual example; avoids full-DataFrame token-id RAM spike."""\n    def __init__(self,df,tokenizer,max_len=128):\n        self.texts=df.text.astype(str).tolist()\n        self.y=df.label.astype(int).tolist()\n        self.tokenizer=tokenizer\n        self.max_len=max_len\n    def __len__(self):return len(self.y)\n    def __getitem__(self,i):\n        enc=self.tokenizer(self.texts[i],truncation=True,max_length=self.max_len)\n        return {**enc,\'labels\':self.y[i]}\n\ndef softmax(logits):\n    z=np.asarray(logits[0] if isinstance(logits,tuple) else logits,dtype=float)\n    z=z-z.max(axis=1,keepdims=True);z=np.exp(z)\n    return z/z.sum(axis=1,keepdims=True)\n\ndef hf_predict(model,tokenizer,df,device,batch_size=16,max_len=128):\n    # Avoid Trainer API/incompatible checkpoint usage during inference; direct torch inference.\n    model.eval();prob=[]\n    with torch.no_grad():\n        for lo in range(0,len(df),batch_size):\n            texts=df.text.iloc[lo:lo+batch_size].astype(str).tolist()\n            inp=tokenizer(texts,truncation=True,max_length=max_len,padding=True,return_tensors=\'pt\')\n            logits=model(**{k:v.to(device) for k,v in inp.items()}).logits\n            prob.append(torch.softmax(logits,dim=1).cpu().numpy())\n    return np.vstack(prob)\n\ndef fit_transformer(train,val,task,name,path,device,epochs=3,batch=4,max_len=128,seed=SEED):\n    AutoTok,AutoModel,Trainer,Args,Collator,last_ckpt=_hf_imports()\n    path=Path(path);path.mkdir(parents=True,exist_ok=True)\n    final=path/\'final\';run=path/\'checkpoints\'\n    base=TRANSFORMERS[name]\n    if (final/\'config.json\').exists():\n        print(\'Loading existing finished transformer:\',task,name)\n        return AutoModel.from_pretrained(str(final)).to(device),AutoTok.from_pretrained(str(final))\n    tokenizer=AutoTok.from_pretrained(base,use_fast=True)\n    model=AutoModel.from_pretrained(base,num_labels=len(LABELS[task]),ignore_mismatched_sizes=True,\n             id2label={i:v for i,v in enumerate(LABELS[task])},\n             label2id={v:i for i,v in enumerate(LABELS[task])})\n    kwargs=dict(output_dir=str(run),num_train_epochs=epochs,learning_rate=2e-5,\n                per_device_train_batch_size=batch,per_device_eval_batch_size=batch*2,\n                gradient_accumulation_steps=4,weight_decay=.01,\n                save_strategy=\'epoch\',load_best_model_at_end=True,metric_for_best_model=\'macro_f1\',\n                greater_is_better=True,save_total_limit=2,report_to=\'none\',seed=seed,\n                fp16=(device.type==\'cuda\'),logging_strategy=\'epoch\',dataloader_num_workers=0)\n    fields=inspect.signature(Args.__init__).parameters\n    kwargs[\'eval_strategy\' if \'eval_strategy\' in fields else \'evaluation_strategy\']=\'epoch\'\n    args=Args(**kwargs)\n    def compute_metrics(ev):\n        pred=ev.predictions[0] if isinstance(ev.predictions,tuple) else ev.predictions\n        return {\'macro_f1\':f1_score(ev.label_ids,np.asarray(pred).argmax(1),average=\'macro\',zero_division=0)}\n    trainer=Trainer(model=model,args=args,\n          train_dataset=TokenData(train,tokenizer,max_len),eval_dataset=TokenData(val,tokenizer,max_len),\n          data_collator=Collator(tokenizer=tokenizer),compute_metrics=compute_metrics)\n    resume=last_ckpt(str(run)) if run.is_dir() else None\n    trainer.train(resume_from_checkpoint=resume)\n    final.mkdir(parents=True,exist_ok=True)\n    trainer.save_model(str(final));tokenizer.save_pretrained(str(final))\n    del trainer;gc.collect()\n    return model.to(device),tokenizer\n\ndef best_threshold(y,prob):\n    y=np.asarray(y,dtype=int);pos=np.asarray(prob)[:,1]\n    candidates=np.round(np.linspace(.10,.90,81),2)\n    s=[f1_score(y,(pos>=t).astype(int),average=\'macro\',zero_division=0) for t in candidates]\n    rank=sorted(range(len(candidates)),key=lambda i:(-s[i],abs(candidates[i]-.5)))\n    return float(candidates[rank[0]])\n\ndef classify(probs,threshold=.5):\n    p=np.asarray(probs)\n    return (p[:,1]>=threshold).astype(int) if p.shape[1]==2 else p.argmax(1)\n\ndef scores(y,probs,task,threshold=.5):\n    true=np.asarray(y,dtype=int);p=np.asarray(probs,dtype=float);pred=classify(p,threshold)\n    vals={\'n\':int(len(true)),\'accuracy\':float(accuracy_score(true,pred)),\n          \'macro_f1\':float(f1_score(true,pred,average=\'macro\',zero_division=0)),\n          \'weighted_f1\':float(f1_score(true,pred,average=\'weighted\',zero_division=0)),\n          \'macro_precision\':float(precision_score(true,pred,average=\'macro\',zero_division=0)),\n          \'macro_recall\':float(recall_score(true,pred,average=\'macro\',zero_division=0)),\n          \'balanced_accuracy\':float(balanced_accuracy_score(true,pred)),\n          \'threshold\':float(threshold)}\n    if p.shape[1]==2:\n        vals.update({\'positive_precision\':float(precision_score(true,pred,zero_division=0)),\n                     \'positive_recall\':float(recall_score(true,pred,zero_division=0)),\n                     \'pr_auc_average_precision\':float(average_precision_score(true,p[:,1])) if len(set(true))==2 else None,\n                     \'brier\':float(brier_score_loss(true,p[:,1]))})\n    return vals\n\ndef model_meta(task,name,data_sig,seed,epochs_lstm,epochs_hf,max_len):\n    return {\'task\':task,\'model\':name,\'split_sha256\':data_sig,\'seed\':seed,\n            \'epochs_lstm\':epochs_lstm,\'epochs_hf\':epochs_hf,\'max_len\':max_len,\n            \'architecture_version\':\'v5.0\'}\n\ndef _check_existing(path,meta):\n    p=Path(path)/\'meta.json\'\n    if p.exists() and json.loads(p.read_text(encoding=\'utf8\'))!=meta:\n        raise RuntimeError(f\'Config/data changed for {path}. Choose a NEW ROOT for a new study, \'\n                           \'not old checkpoints from a different dataset or configuration.\')\n    path.mkdir(parents=True,exist_ok=True)\n    if not p.exists():p.write_text(json.dumps(meta,indent=2),encoding=\'utf8\')\n\ndef _device():return torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n\ndef load_model(root,task,name,device=None):\n    root=Path(root);device=device or _device();path=root/\'models\'/task/name\n    if name==\'ensemble2\':\n        obj=json.loads((path/\'ensemble.json\').read_text(encoding=\'utf8\'))\n        members=[load_model(root,task,m,device) for m in obj[\'members\']]\n        return (\'ensemble\',members,obj[\'weights\'])\n    if name.startswith(\'tfidf\'):\n        return (\'classical\',joblib.load(path/\'model.joblib\'),None)\n    if name.startswith(\'bilstm\'):\n        vocab=json.loads((path/\'vocab.json\').read_text(encoding=\'utf8\'))\n        model=BiLSTM(len(vocab),len(LABELS[task]),pool=name.split(\'_\',1)[1]).to(device)\n        model.load_state_dict(torch.load(path/\'best.pt\',map_location=device,weights_only=True))\n        return (\'neural\',model,vocab)\n    # Deployment needs only Auto classes, not Trainer or Accelerate.\n    from transformers import AutoTokenizer as AutoTok, AutoModelForSequenceClassification as AutoModel\n    model=AutoModel.from_pretrained(str(path/\'final\')).to(device)\n    tok=AutoTok.from_pretrained(str(path/\'final\'))\n    return (\'transformer\',model,tok)\n\ndef predict_loaded(loaded,df,device=None,max_len=128):\n    device=device or _device();typ,model,aux=loaded\n    if typ==\'ensemble\':\n        return sum(float(w)*predict_loaded(member,df,device,max_len=max_len) for w,member in zip(aux,model))\n    if typ==\'classical\':return model.predict_proba(df.text.astype(str))\n    if typ==\'neural\':return neural_predict(model,df,aux,device)\n    return hf_predict(model,aux,df,device,max_len=max_len)\n\ndef fit_one(root,frames,task,name,seed=SEED,epochs_lstm=8,epochs_hf=3,max_len=128):\n    assert name in MODEL_NAMES,f\'Unknown model: {name}\'\n    root=output_paths(root);sig=fingerprint(frames);path=root/\'models\'/task/name\n    meta=model_meta(task,name,sig,seed,epochs_lstm,epochs_hf,max_len)\n    _check_existing(path,meta)\n    done=path/\'validated.json\';device=_device()\n    if done.exists():\n        record=json.loads(done.read_text(encoding=\'utf8\'))\n        print(f\'SKIP completed {task}/{name} (val macro-F1={record["metrics"]["macro_f1"]:.4f})\')\n        return record\n    set_seed(seed)\n    train,val=frames[\'train\'],frames[\'val\']\n    if name.startswith(\'tfidf\'):\n        model=feature_model(name);model.fit(train.text.astype(str),train.label)\n        joblib.dump(model,path/\'model.joblib\',compress=3)\n        loaded=(\'classical\',model,None)\n    elif name.startswith(\'bilstm\'):\n        net,vocab=fit_bilstm(train,val,task,name,path,device,epochs_lstm,seed=seed)\n        loaded=(\'neural\',net,vocab)\n    else:\n        model,tokenizer=fit_transformer(train,val,task,name,path,device,epochs_hf,max_len=max_len,seed=seed)\n        loaded=(\'transformer\',model,tokenizer)\n    prob=predict_loaded(loaded,val,device,max_len)\n    threshold=best_threshold(val.label,prob) if len(LABELS[task])==2 else .5\n    metrics=scores(val.label,prob,task,threshold)\n    np.savez_compressed(path/\'validation_probabilities.npz\',probs=prob,labels=val.label.to_numpy(dtype=int))\n    # Review criterion is validation-derived quantile, NOT claimed calibrated probability.\n    review_cutoff=float(np.quantile(np.max(prob,axis=1),.20))\n    record={\'meta\':meta,\'metrics\':metrics,\'review_below\':review_cutoff,\'source_split\':\'val\'}\n    done.write_text(json.dumps(record,indent=2),encoding=\'utf8\')\n    print(f\'COMPLETED {task}/{name}: validation macro-F1={metrics["macro_f1"]:.4f}, threshold={threshold:.2f}\')\n    del loaded;gc.collect()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n    return record\n\ndef scoreboard(root,task):\n    root=Path(root);rows=[]\n    for name in [*MODEL_NAMES,\'ensemble2\']:\n        p=root/\'models\'/task/name/\'validated.json\'\n        if p.exists():\n            r=json.loads(p.read_text());rows.append({\'model\':name,**r[\'metrics\']})\n    if not rows:return pd.DataFrame()\n    out=pd.DataFrame(rows).sort_values([\'macro_f1\',\'model\'],ascending=[False,True]).reset_index(drop=True)\n    out.to_csv(root/\'results\'/f\'{task}_validation_scores.csv\',index=False)\n    return out\n\ndef propose_validation_ensemble(root,task,min_gain=.01):\n    """Validation-only 2-model mixture; require >1pp macro-F1 improvement to avoid noise chasing.\n\n    Call before locking; the test is neither loaded nor read here.\n    """\n    root=Path(root)\n    tab=scoreboard(root,task)\n    singles=tab[tab.model!=\'ensemble2\']\n    if len(singles)<2:return {\'status\':\'not_enough_completed_models\'}\n    names=singles.model.iloc[:2].tolist()\n    loaded=[np.load(root/\'models\'/task/n/\'validation_probabilities.npz\') for n in names]\n    p1,p2=(a[\'probs\'] for a in loaded)\n    y1,y2=(a[\'labels\'] for a in loaded)\n    if not np.array_equal(y1,y2):raise RuntimeError(\'Validation probability label sequences mismatch\')\n    vals=[]\n    for w in np.linspace(.2,.8,7):\n        mix=w*p1+(1-w)*p2\n        threshold=best_threshold(y1,mix) if mix.shape[1]==2 else .5\n        metric=scores(y1,mix,task,threshold)\n        vals.append((metric[\'macro_f1\'],float(w),float(threshold),metric,mix))\n    chosen=max(vals,key=lambda z:z[0])\n    gain=chosen[0]-float(singles.iloc[0].macro_f1)\n    if gain<min_gain:\n        return {\'status\':\'rejected\',\'members\':names,\'best_gain\':round(float(gain),5),\'required_gain\':min_gain}\n    path=root/\'models\'/task/\'ensemble2\'\n    if (root/\'results\'/\'locked_validation_winners.json\').exists():\n        locked=json.loads((root/\'results\'/\'locked_validation_winners.json\').read_text())\n        if task in locked:raise RuntimeError(\'Refusing to create validation ensemble after task lock\')\n    path.mkdir(parents=True,exist_ok=True)\n    sig=json.loads((root/\'models\'/task/names[0]/\'validated.json\').read_text())[\'meta\'][\'split_sha256\']\n    for name in names:\n        meta=json.loads((root/\'models\'/task/name/\'validated.json\').read_text())[\'meta\']\n        if meta[\'split_sha256\']!=sig:raise RuntimeError(\'Members trained on different splits\')\n    path.joinpath(\'ensemble.json\').write_text(json.dumps({\'members\':names,\'weights\':[chosen[1],1-chosen[1]]},indent=2))\n    meta={\'task\':task,\'model\':\'ensemble2\',\'split_sha256\':sig,\'max_len\':128,\n          \'architecture_version\':\'v5.0\',\'ensemble_members\':names}\n    result={\'meta\':meta,\'metrics\':chosen[3],\n            \'review_below\':float(np.quantile(chosen[4].max(axis=1),.20)), \'source_split\':\'val\'}\n    path.joinpath(\'validated.json\').write_text(json.dumps(result,indent=2))\n    np.savez_compressed(path/\'validation_probabilities.npz\',probs=chosen[4],labels=y1)\n    return {\'status\':\'accepted\',\'members\':names,\'gain\':float(gain),\'weights\':[chosen[1],1-chosen[1]],\'metrics\':chosen[3]}\n\ndef select_and_lock(root,data,tasks,allow_existing=True):\n    root=Path(root);lock_file=root/\'results\'/\'locked_validation_winners.json\'\n    if lock_file.exists():\n        old=json.loads(lock_file.read_text(encoding=\'utf8\'))\n    else:old={}\n    for task in tasks:\n        if task in old:\n            print(f\'{task} ALREADY LOCKED -> {old[task]["model"]}. No reselection permitted.\')\n            continue\n        tab=scoreboard(root,task)\n        if tab.empty:raise RuntimeError(f\'No models trained for {task}\')\n        winner=tab.iloc[0][\'model\'];record=json.loads((root/\'models\'/task/winner/\'validated.json\').read_text())\n        if record[\'meta\'][\'split_sha256\']!=fingerprint(data[task]):raise RuntimeError(\'Dataset fingerprint mismatch\')\n        old[task]={\'model\':winner,\'threshold\':record[\'metrics\'][\'threshold\'],\n                   \'review_below\':record[\'review_below\'], \'split_sha256\':record[\'meta\'][\'split_sha256\'],\n                   \'selected_on\':\'validation macro-F1; holdout unseen\', \'candidate_count\':len(tab)}\n    lock_file.write_text(json.dumps(old,indent=2),encoding=\'utf8\')\n    return old\n\ndef test_locked(root,frames,task,save_private_errors=True):\n    root=Path(root);f=root/\'results\'/\'locked_validation_winners.json\'\n    if not f.exists():raise RuntimeError(\'Select & lock winner BEFORE testing\')\n    r=json.loads(f.read_text(encoding=\'utf8\'))\n    if task not in r:raise RuntimeError(f\'{task} not locked\')\n    locked=r[task]\n    if locked[\'split_sha256\']!=fingerprint(frames):raise RuntimeError(\'Test data signature differs from locked study\')\n    path=root/\'results\'/f\'{task}_locked_test.json\'\n    if path.exists():\n        print(f\'{task} holdout already evaluated: leaving existing result untouched\')\n        return json.loads(path.read_text())\n    test=frames[\'test\'];device=_device()\n    metadata=json.loads((root/\'models\'/task/locked[\'model\']/\'validated.json\').read_text())[\'meta\']\n    prob=predict_loaded(load_model(root,task,locked[\'model\'],device),test,device,max_len=metadata[\'max_len\'])\n    y=np.asarray(test.label)\n    preds=classify(prob,locked[\'threshold\'])\n    report={\'task\':task,\'winner\':locked[\'model\'], \'source\':(\'previously_exposed_v4_internal_holdout\' if task==\'sentiment\' else\n                      \'team_mirror_sentiMix_holdout_not_official_test\' if task==\'sentiment_raw\' else \'source_supplied_holdout\'),\n            \'metrics\':scores(y,prob,task,locked[\'threshold\']),\n            \'confusion\':confusion_matrix(y,preds,labels=range(len(LABELS[task]))).tolist(),\n            \'per_class\':classification_report(y,preds,labels=range(len(LABELS[task])),\n                          target_names=LABELS[task],zero_division=0,output_dict=True)}\n    err=test.loc[preds!=y,[\'text\',\'label\']].copy()\n    err[\'pred\']=preds[preds!=y]\n    err[\'id\']=err.text.map(lambda x:hashlib.sha256(str(x).encode()).hexdigest()[:14])\n    if save_private_errors:err.to_csv(root/\'private_analysis\'/f\'{task}_test_errors_PRIVATE.csv\',index=False)\n    # Never include raw training/test tweets in the public metric JSON.\n    path.write_text(json.dumps(report,indent=2),encoding=\'utf8\')\n    return report\n\ndef synthetic_perturb(text):\n    alias={\'bahut\':\'bohot\',\'bohot\':\'bahut\',\'acha\':\'accha\',\'accha\':\'acha\',\n           \'nahi\':\'nhi\',\'nhi\':\'nahi\',\'yaar\':\'yr\',\'kyun\':\'kyu\',\'pyar\':\'pyaar\'}\n    return re.sub(r\'(?ui)\\b\\w+\\b\',lambda m:alias.get(m.group(0).lower(),m.group(0)),str(text))\n\ndef script_bucket(text):\n    # Script categories are NOT language IDs; Latin-only includes both English and Roman Hinglish.\n    t=str(text)\n    deva=bool(re.search(r\'[\\u0900-\\u097F]\',t))\n    latin=bool(re.search(r\'[A-Za-z]\',t))\n    return (\'mixed_devanagari_latin\' if deva and latin else\n            \'devanagari_only\' if deva else \'latin_script\' if latin else \'other\')\n\ndef robustness_locked(root,frames,task):\n    root=Path(root);lock=json.loads((root/\'results\'/\'locked_validation_winners.json\').read_text())[task]\n    if fingerprint(frames)!=lock[\'split_sha256\']:raise ValueError(\'Fingerprint mismatch\')\n    test=frames[\'test\'];device=_device();loader=load_model(root,task,lock[\'model\'],device)\n    metadata=json.loads((root/\'models\'/task/lock[\'model\']/\'validated.json\').read_text())[\'meta\']\n    max_len=metadata[\'max_len\']\n    altered=test.copy();altered[\'text\']=altered.text.map(synthetic_perturb)\n    changed=(altered.text!=test.text)\n    result={\'task\':task,\'method\':\'synthetic spelling variants; NOT a human-verified robustness benchmark\',\n            \'n_changed\':int(changed.sum())}\n    if changed.sum()>10:\n        a=predict_loaded(loader,test.loc[changed].reset_index(drop=True),device,max_len=max_len)\n        b=predict_loaded(loader,altered.loc[changed].reset_index(drop=True),device,max_len=max_len)\n        y=test.loc[changed].label\n        result[\'original\']=scores(y,a,task,lock[\'threshold\'])\n        result[\'perturbed\']=scores(y,b,task,lock[\'threshold\'])\n        result[\'macro_f1_delta\']=result[\'perturbed\'][\'macro_f1\']-result[\'original\'][\'macro_f1\']\n    result[\'script_slices\']={}\n    for sc in sorted(set(test.text.map(script_bucket))):\n        subset=test[test.text.map(script_bucket)==sc]\n        if len(subset)<20 or subset.label.nunique()<2:continue\n        px=predict_loaded(loader,subset,device,max_len=max_len)\n        result[\'script_slices\'][sc]=scores(subset.label,px,task,lock[\'threshold\'])\n    result[\'script_slices_note\']=\'Script buckets are NOT validated language labels; Latin includes English/Hinglish.\'\n    if task==\'hate\' and \'lang\' in test.columns:\n        result[\'language_slices\']={}\n        for lang,g in test.groupby(\'lang\'):\n            if len(g)<10:continue\n            p=predict_loaded(loader,g,device,max_len=max_len)\n            result[\'language_slices\'][str(lang)]=scores(g.label,p,task,lock[\'threshold\'])\n    (root/\'results\'/f\'{task}_robustness.json\').write_text(json.dumps(result,indent=2),encoding=\'utf8\')\n    return result\n\ndef predict_text(root,task,text,loaded=None,device=None):\n    root=Path(root);device=device or _device()\n    lock_file=root/\'results\'/\'locked_validation_winners.json\'\n    locks=json.loads(lock_file.read_text(encoding=\'utf8\'))\n    if task not in locks:raise RuntimeError(f\'{task} has no locked trained model\')\n    spec=locks[task];loaded=loaded or load_model(root,task,spec[\'model\'],device)\n    dummy=pd.DataFrame({\'text\':[clean(text)],\'label\':[0]})\n    metadata=json.loads((root/\'models\'/task/spec[\'model\']/\'validated.json\').read_text())[\'meta\']\n    p=predict_loaded(loaded,dummy,device,max_len=metadata[\'max_len\'])[0]\n    idx=int(classify(p[None,:],spec[\'threshold\'])[0])\n    return {\'task\':task,\'model\':spec[\'model\'],\'label\':LABELS[task][idx],\n            \'probabilities\':{name:round(float(v),4) for name,v in zip(LABELS[task],p)},\n            \'uncalibrated_max_probability\':round(float(np.max(p)),4),\n            \'review_required\':bool(np.max(p)<spec[\'review_below\']),\n            \'review_threshold_validation_derived\':round(float(spec[\'review_below\']),4),\n            \'positive_decision_threshold\':spec[\'threshold\'] if len(p)==2 else None,\n            \'calibrated\':False,\'research_only\':True}\n\n\ndef paired_bootstrap_validation(root,task,model_a,model_b,repeats=1000,seed=SEED):\n    """Paired bootstrap of *validation* macro-F1 difference B - A; not a test-set significance claim."""\n    root=Path(root)\n    vals=[]\n    for name in (model_a,model_b):\n        p=root/\'models\'/task/name/\'validation_probabilities.npz\'\n        if not p.exists():raise FileNotFoundError(p)\n        arr=np.load(p)\n        params=json.loads((root/\'models\'/task/name/\'validated.json\').read_text())[\'metrics\']\n        vals.append((arr[\'probs\'],arr[\'labels\'],float(params[\'threshold\'])))\n    (pa,ya,ta),(pb,yb,tb)=vals\n    if not np.array_equal(ya,yb):raise ValueError(\'Validation samples differ; comparison invalid\')\n    a=classify(pa,ta);b=classify(pb,tb);rng=np.random.default_rng(seed)\n    n=len(ya);d=[]\n    for _ in range(repeats):\n        indices=rng.integers(0,n,n)\n        d.append(f1_score(ya[indices],b[indices],average=\'macro\',zero_division=0)-\n                 f1_score(ya[indices],a[indices],average=\'macro\',zero_division=0))\n    point=float(f1_score(ya,b,average=\'macro\')-f1_score(ya,a,average=\'macro\'))\n    return {\'task\':task,\'model_a\':model_a,\'model_b\':model_b,\n            \'delta_b_minus_a\':point,\'bootstrap_95_pct_interval\':list(map(float,np.quantile(d,[.025,.975]))),\n            \'samples\':n,\'bootstrap_repeats\':repeats,\'scope\':\'same validation examples; exploratory\'}\n\n\ndef evaluate_external_challenge(root,task,path):\n    """Independently annotated challenge CSV with exact, task-specific human labels.\n\n    Reject any text seen in *any* experiment split. Does not edit/select models.\n    """\n    root=Path(root)\n    csv=Path(path)\n    df=pd.read_csv(csv)\n    if not {\'text\',\'label\'}.issubset(df.columns):\n        raise ValueError(\'Challenge CSV needs text,label. Label must be an exact task label name.\')\n    if df.text.isna().any() or df.label.isna().any():raise ValueError(\'Challenge labels cannot be blank\')\n    df[\'text\']=df.text.astype(str).map(clean)\n    df[\'label\']=df.label.astype(str).str.strip()\n    classes=LABELS[task]\n    if not set(df.label).issubset(set(classes)):\n        raise ValueError(f\'Challenge labels must be one of: {classes}. Got: {sorted(set(df.label)-set(classes))}\')\n    if len(df)<20:raise ValueError(\'Challenge set too small; require >=20 human-labeled examples\')\n    keys=_key_series(df)\n    if keys.duplicated().any():raise ValueError(\'Challenge set contains exact duplicate texts\')\n    seen=set()\n    for split in (\'train\',\'val\',\'test\'):\n        dftrain=pd.read_csv(root/\'data\'/f\'{task}_{split}.csv\')\n        seen.update(_key_series(dftrain))\n    overlap=set(keys)&seen\n    if overlap:raise ValueError(f\'{len(overlap)} challenge texts overlap training/validation/test; refusing contaminated evaluation\')\n    locked=json.loads((root/\'results\'/\'locked_validation_winners.json\').read_text())\n    if task not in locked:raise RuntimeError(\'Task must be locked before independent evaluation\')\n    info=locked[task]\n    metadata=json.loads((root/\'models\'/task/info[\'model\']/\'validated.json\').read_text())[\'meta\']\n    tmp=df[[\'text\']].copy();tmp[\'label\']=df.label.map({name:i for i,name in enumerate(classes)})\n    probs=predict_loaded(load_model(root,task,info[\'model\']),tmp,max_len=metadata[\'max_len\'])\n    result={\'source\':\'human_annotated_independent_challenge\',\n            \'task\':task,\'model\':info[\'model\'],\n            \'metrics\':scores(tmp.label,probs,task,info[\'threshold\']),\n            \'per_class\':classification_report(tmp.label,classify(probs,info[\'threshold\']),\n                        labels=range(len(classes)),target_names=classes,output_dict=True,zero_division=0),\n            \'scope\':\'Human annotation quality and independence must be documented separately\'}\n    out=root/\'results\'/f\'{task}_external_challenge.json\'\n    out.write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding=\'utf8\')\n    return result\n'
import subprocess
PROJECT_CHECKOUT=Path('/content/MixBhasha_v5_latest')
if not PROJECT_CHECKOUT.exists():
    print('Fetching current MixBhasha source from GitHub...',flush=True)
    subprocess.run(['git','clone','--depth','1','https://github.com/RezZup/MixBhasha_git.git',str(PROJECT_CHECKOUT)],check=True)
else:
    subprocess.run(['git','-C',str(PROJECT_CHECKOUT),'pull','--ff-only','origin','main'],check=True)
CORE_CODE=(PROJECT_CHECKOUT/'src'/'mixbhasha_core.py').read_text(encoding='utf8')
Path('mixbhasha_core.py').write_text(CORE_CODE,encoding='utf8')
import importlib, mixbhasha_core as core
importlib.reload(core)
print('Models available:',core.MODEL_NAMES)
print('Tasks:',core.LABELS)

Fetching current MixBhasha source from GitHub...
Models available: ['tfidf_word', 'tfidf_char', 'tfidf_word_char', 'tfidf_normalized', 'tfidf_svm', 'bilstm_last', 'bilstm_meanmax', 'bilstm_attention', 'distilbert', 'muril', 'hingbert', 'xlmr', 'hing_roberta_mixed', 'twitter_xlmr', 'twitter_xlmr_sentiment']
Tasks: {'sentiment': ['negative', 'neutral', 'positive'], 'sentiment_raw': ['negative', 'neutral', 'positive'], 'cyberbullying7': ['not_cyberbullying', 'age', 'gender', 'religion', 'mockery', 'abusive', 'offensive'], 'cyberbullying_binary': ['not_cyberbullying', 'cyberbullying_category'], 'offense_hasoc': ['not_hate_offensive', 'hate_or_offensive'], 'hate': ['non_hate', 'hate']}


## 3 — Acquire sources, clean and freeze disjoint partitions

**Dataset tasks:**

- `sentiment`: third-party processed SentiMix, for a processed-source ablation only; v5 changes row cleaning vs v4, and the dev holdout was previously exposed.
- `sentiment_raw`: *different* BAKSA team mirror of SentiMix raw Hinglish tweets with 3 class labels. This restores original-style spelling and mixed scripts, but is NOT independently verified official leaderboard test. Scores are **not directly comparable** with v4 and may share examples with the processed derivative.
- `cyberbullying7`: all seven MC-Hinglish1.0 classes, no collapsed/excluded categories.
- `cyberbullying_binary`: derived **any of six labeled harm categories vs not-cyberbullying**, not necessarily all offensive/hate speech.
- `offense_hasoc`: manually supplied authorized HOF/NOT, may need parent context.
- `hate`: separate PRISM task, **hate != offense**.

**Leakage policy:** train-only contradictions are dropped; exact source-test text overlap is removed from the training pool **without accessing test labels**. Original source test rows remain in place (only non-text/invalid label rows removed). Once trained, changing prepared splits requires a new experiment root.

In [4]:
print('Experiment root:',ROOT,flush=True)
audit_path=ROOT/'results'/'data_audit.json'
saved_audit=json.loads(audit_path.read_text(encoding='utf8')) if audit_path.is_file() else {}
DATA={};AUDITS={};missing_tasks=[]
for task in RUN_TASKS:
    split_paths=[ROOT/'data'/f'{task}_{split}.csv' for split in ('train','val','test')]
    if all(path.is_file() for path in split_paths) and task in saved_audit:
        cached=core.load_prepared(ROOT,[task])[task]
        actual_fingerprint=core.fingerprint(cached)
        expected_fingerprint=saved_audit[task].get('split_sha256')
        if actual_fingerprint!=expected_fingerprint:
            raise RuntimeError(f'{task}: Drive split files do not match data_audit.json; refusing to overwrite or train.')
        DATA[task]=cached;AUDITS[task]=saved_audit[task]
        print(f'Using cached {task} splits (fingerprint verified); no data preparation repeated.',flush=True)
    else:
        missing_tasks.append(task)
if missing_tasks and not PREPARE_MISSING_TASKS:
    raise FileNotFoundError(f'Missing prepared tasks in {ROOT}: {missing_tasks}. Check Drive mount/path. '
        'Set PREPARE_MISSING_TASKS=True only when you intend to prepare data; existing model fingerprints are checked before writing.')
if missing_tasks:
    print('Preparing explicitly requested missing tasks:',missing_tasks,flush=True)
    prepared,prepared_audits=core.prepare(ROOT,tasks=missing_tasks,hasoc_train=HASOC_TRAIN_CSV,
        hasoc_test=HASOC_TEST_CSV,mc_label_map=MC_LABEL_MAP,
        mc_text_column=MC_TEXT_COLUMN,mc_label_column=MC_LABEL_COLUMN)
    DATA.update(prepared);AUDITS.update(prepared_audits)
for task in RUN_TASKS:
    print('\nTASK:',task,'|',AUDITS[task]['source'])
    print('SHA256:',AUDITS[task]['split_sha256'])
    display(pd.DataFrame({k:d.label.value_counts().sort_index() for k,d in DATA[task].items()}).fillna(0).astype(int))
    print('Existing split audit retained at:',audit_path)

Experiment root: /content/drive/MyDrive/MixBhasha_V5_seed42
Using cached cyberbullying7 splits (fingerprint verified); no data preparation repeated.
Using cached sentiment_raw splits (fingerprint verified); no data preparation repeated.

TASK: cyberbullying7 | MC-Hinglish1.0 all 7 categories; derived binary supported but separately named
SHA256: 9f77c51d6f354a3aa6421325cac8f67992421d2c253fb9acd8b44c7922945c50


,train,val,test
label,,,
0,1298,177,200
1,849,116,129
2,876,120,141
3,854,116,131
4,861,117,127
5,858,117,148
6,849,116,124


Existing split audit retained at: /content/drive/MyDrive/MixBhasha_V5_seed42/results/data_audit.json

TASK: sentiment_raw | BAKSA public team mirror of SentiMix Hinglish raw text; NOT organizer-verified official leaderboard test
SHA256: 77e4b2a60b872ce79c590e3bf04c80128fb5f0a80134f12abc95faf96b6c970e


,train,val,test
label,,,
0,2917,398,979
1,3797,518,1210
2,3419,466,941


Existing split audit retained at: /content/drive/MyDrive/MixBhasha_V5_seed42/results/data_audit.json


## 4 — Split-integrity checks and dataset coverage

In [5]:
for task,frames in DATA.items():
    keys={k:set(core._key_series(df)) for k,df in frames.items()}
    assert not (keys['train']&keys['val'] or keys['train']&keys['test'] or keys['val']&keys['test'])
    assert all(df.label.nunique()==len(core.LABELS[task]) for df in frames.values())
    for split in ('train','val','test'):
        counts=frames[split].label.value_counts().sort_index()
        print(task,split,len(frames[split]),counts.to_dict())
print('PASS: no exact-text overlap and all classes represented.')

cyberbullying7 train 6445 {0: 1298, 1: 849, 2: 876, 3: 854, 4: 861, 5: 858, 6: 849}
cyberbullying7 val 879 {0: 177, 1: 116, 2: 120, 3: 116, 4: 117, 5: 117, 6: 116}
cyberbullying7 test 1000 {0: 200, 1: 129, 2: 141, 3: 131, 4: 127, 5: 148, 6: 124}
sentiment_raw train 10133 {0: 2917, 1: 3797, 2: 3419}
sentiment_raw val 1382 {0: 398, 1: 518, 2: 466}
sentiment_raw test 3130 {0: 979, 1: 1210, 2: 941}
PASS: no exact-text overlap and all classes represented.


## Training data quality audit (no retraining)

Run after preparing `cyberbullying7` splits and before model training. The audit reads training labels, but reads only the text column from validation/test CSVs. Aggregate outputs contain no raw examples; row-level review candidates go to `private_analysis/`.


In [6]:
from pathlib import Path
import sys

AUDIT_ROOT = Path(ROOT)
AUDIT_CHECKOUT = PROJECT_CHECKOUT
AUDIT_SCRIPT = AUDIT_CHECKOUT / 'tools' / 'audit_training_data.py'

# Clone the project so this cell can run directly in Colab after the audit
# script has been pushed to the repository. No model-training code is called.
if not AUDIT_SCRIPT.is_file():
    if not AUDIT_CHECKOUT.exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/RezZup/MixBhasha_git.git', str(AUDIT_CHECKOUT)],
            check=True,
        )
if not AUDIT_SCRIPT.is_file():
    raise FileNotFoundError(
        f'Audit script not found at {AUDIT_SCRIPT}. Confirm the latest project '
        'revision is available in GitHub and that the repository checkout succeeded.'
    )
required = [
    AUDIT_ROOT / 'data' / f'cyberbullying7_{split}.csv'
    for split in ('train', 'val', 'test')
]
required.append(AUDIT_ROOT / 'results' / 'data_audit.json')
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'Missing prepared cyberbullying7 files: ' + ', '.join(missing) +
        '. Run the existing V5 data-preparation cell with RUN_TASKS containing '
        "'cyberbullying7' first; this audit will not prepare or modify data."
    )
subprocess.run(
    [sys.executable, str(AUDIT_SCRIPT), '--root', str(AUDIT_ROOT)],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '/content/MixBhasha_v5_latest/tools/audit_training_data.py', '--root', '/content/drive/MyDrive/MixBhasha_V5_seed42'], returncode=0)

## Cyberbullying7 SVM optimization (training-only CV)

Run this cell separately after mounting Drive. It reads the existing train/validation splits and the saved `tfidf_svm` baseline, never reads the test split, and does not run the notebook's six-model training loop. The nested five-fold search covers 48 feature/C/class-weight configurations with three-fold calibration; progress is printed by feature and CV fold, but it can take a while. Existing optimization outputs are protected unless `OVERWRITE_EXISTING_OPTIMIZATION` is explicitly enabled.


In [7]:
from pathlib import Path
import subprocess
import sys

if globals().get('FINALIZE', False):
    raise RuntimeError('Set FINALIZE=False before optimization; test evaluation is not part of this experiment.')

OPTIMIZATION_ROOT = Path(ROOT)
OPTIMIZER_SCRIPT = PROJECT_CHECKOUT / 'tools' / 'optimize_svm.py'
OVERWRITE_EXISTING_OPTIMIZATION = False

if not OPTIMIZER_SCRIPT.is_file():
    raise FileNotFoundError(f'Optimizer source not found: {OPTIMIZER_SCRIPT}')
if not (OPTIMIZATION_ROOT / 'data' / 'cyberbullying7_train.csv').is_file():
    raise FileNotFoundError(f'Prepared cyberbullying7 split is missing under {OPTIMIZATION_ROOT}.')
sys.path.insert(0, str(PROJECT_CHECKOUT / 'tools'))
import optimize_svm
print('Starting SVM optimization; CV progress and ETA will stream below.', flush=True)
OPTIMIZATION_REPORT = optimize_svm.run_experiment(
    OPTIMIZATION_ROOT,
    overwrite=OVERWRITE_EXISTING_OPTIMIZATION,
)


Starting SVM optimization; CV progress and ETA will stream below.
[1/5] Loading prepared train/validation splits from /content/drive/MyDrive/MixBhasha_V5_seed42
REUSE completed SVM optimization from /content/drive/MyDrive/MixBhasha_V5_seed42/experiments/cyberbullying7_svm_optimization; no CV work repeated. Set overwrite=True only to intentionally rerun.


## Cyberbullying7 fixed ensemble evaluation

Run this cell separately after the existing validation predictions and model checkpoints are present in Drive. It verifies fingerprints, saved validation-label order, probability arrays, and class mappings without rerunning HingRoBERTa inference. It does not retrain models or load test data. Only the prespecified 50/50, 75/25, and SVM-tie-break hard-vote comparisons are evaluated. Outputs go to `experiments/cyberbullying7_ensemble/`.

In [9]:
from pathlib import Path
import json

def detect_all_models(root_dir):
    root_path = Path(root_dir)
    models_dir = root_path / 'models'
    if not models_dir.is_dir():
        print(f'Models directory does not exist at: {models_dir}')
        return

    print(f'Scanning: {models_dir}\n')
    for task_dir in sorted(models_dir.iterdir()):
        if task_dir.is_dir():
            print(f'📌 Task: {task_dir.name}')
            for model_dir in sorted(task_dir.iterdir()):
                if model_dir.is_dir():
                    validated_json = model_dir / 'validated.json'
                    has_meta = validated_json.exists()
                    has_probs = (model_dir / 'validation_probabilities.npz').exists()

                    # Check model type indicators
                    is_tfidf = any(model_dir.glob('*.joblib'))
                    is_bilstm = (model_dir / 'best.pt').exists()
                    is_transformer = (model_dir / 'final' / 'config.json').exists()

                    model_type = 'Unknown'
                    if is_tfidf: model_type = 'TF-IDF / Classical'
                    elif is_bilstm: model_type = 'BiLSTM'
                    elif is_transformer: model_type = 'Transformer'
                    elif model_dir.name == 'ensemble2': model_type = 'Ensemble'

                    status = '✅ Complete' if (has_meta and has_probs) else '❌ Incomplete'
                    print(f'  - Model: {model_dir.name:<25} | Type: {model_type:<20} | Status: {status}')
                    if has_meta:
                        try:
                            meta = json.loads(validated_json.read_text())
                            f1 = meta.get('metrics', {}).get('macro_f1')
                            if f1 is not None:
                                print(f'    └─ Val Macro-F1: {f1:.4f}')
                        except Exception:
                            pass
            print()

detect_all_models(ROOT)

Scanning: /content/drive/MyDrive/MixBhasha_V5_seed42/models

📌 Task: cyberbullying7
  - Model: bilstm_attention          | Type: BiLSTM               | Status: ✅ Complete
    └─ Val Macro-F1: 0.7118
  - Model: hing_roberta_mixed        | Type: Transformer          | Status: ✅ Complete
    └─ Val Macro-F1: 0.7958
  - Model: muril                     | Type: Transformer          | Status: ✅ Complete
    └─ Val Macro-F1: 0.5390
  - Model: tfidf_svm                 | Type: TF-IDF / Classical   | Status: ✅ Complete
    └─ Val Macro-F1: 0.8127
  - Model: tfidf_word_char           | Type: TF-IDF / Classical   | Status: ✅ Complete
    └─ Val Macro-F1: 0.7804
  - Model: twitter_xlmr              | Type: Transformer          | Status: ✅ Complete
    └─ Val Macro-F1: 0.7599

📌 Task: sentiment_raw
  - Model: hing_roberta_mixed        | Type: Transformer          | Status: ✅ Complete
    └─ Val Macro-F1: 0.6684
  - Model: muril                     | Type: Transformer          | Status: ✅ Complete
 

In [12]:
from pathlib import Path

if globals().get('FINALIZE', False):
    raise RuntimeError('Set FINALIZE=False; test evaluation is not part of this ensemble experiment.')

ENSEMBLE_ROOT = Path(ROOT)
ENSEMBLE_SCRIPT = PROJECT_CHECKOUT / 'tools' / 'evaluate_ensemble.py'
OVERWRITE_EXISTING_ENSEMBLE = False

if not ENSEMBLE_SCRIPT.is_file():
    raise FileNotFoundError(
        f'Ensemble evaluator not found at {ENSEMBLE_SCRIPT}. Refresh the Colab checkout '
        'from the latest main branch, then rerun this cell.'
    )
sys.path.insert(0, str(PROJECT_CHECKOUT / 'tools'))
import evaluate_ensemble

print('Starting fast ensemble evaluation; model inference is not rerun.', flush=True)
ENSEMBLE_REPORT = evaluate_ensemble.run_evaluation(
    ENSEMBLE_ROOT,
    overwrite=OVERWRITE_EXISTING_ENSEMBLE,
)


Starting fast ensemble evaluation; model inference is not rerun.
[1/4] Loading validation-only data and stored split fingerprint.
[2/4] Checking model class orders from saved artifacts, without model inference.
[2/4] Checking saved predictions 1/2: tfidf_svm.
Verified tfidf_svm: fingerprint, class order, and saved label sequence.
[2/4] Checking saved predictions 2/2: hing_roberta_mixed.
Verified hing_roberta_mixed: fingerprint, class order, and saved label sequence.
[3/4] Computing fixed ensemble predictions and validation metrics.
Bootstrap ensemble_equal_50_50: 95% CI [-0.016751724136482276, 0.030284834236394234]
Bootstrap ensemble_svm_75_roberta_25: 95% CI [-0.002777163633361698, 0.028833985333255353]
Bootstrap ensemble_hard_vote_svm_tiebreak: 95% CI [0.0, 0.0]
[4/4] Writing tables, reports, and visualizations.
Ensemble evaluation complete in 0.2 min: /content/drive/MyDrive/MixBhasha_V5_seed42/experiments/cyberbullying7_ensemble


In [11]:
import json
from pathlib import Path

# Path to the HingRoBERTa model configuration
config_path = ROOT / 'models' / 'cyberbullying7' / 'hing_roberta_mixed' / 'final' / 'config.json'

if config_path.is_file():
    print(f'Loading config from {config_path}')
    config = json.loads(config_path.read_text())

    # Update config values to match the 7 classes of cyberbullying7
    classes = ['not_cyberbullying', 'age', 'gender', 'religion', 'mockery', 'abusive', 'offensive']
    config['num_labels'] = len(classes)
    config['id2label'] = {str(i): name for i, name in enumerate(classes)}
    config['label2id'] = {name: i for i, name in enumerate(classes)}

    # Save the updated config
    config_path.write_text(json.dumps(config, indent=2))
    print('✅ config.json successfully updated with 7 output labels!')
else:
    print(f'❌ Config file not found at: {config_path}')

Loading config from /content/drive/MyDrive/MixBhasha_V5_seed42/models/cyberbullying7/hing_roberta_mixed/final/config.json
✅ config.json successfully updated with 7 output labels!


## 5 — Train only the chosen models (checkpoint-aware, opt-in)

**Model training is disabled by default.** Completed Drive artifacts are reused; incomplete models are reported but not started unless you set `RUN_MISSING_MODELS=True`. Transformer checkpoints resume when training is interrupted. The cell prints elapsed-time heartbeats and Hugging Face's step progress/ETA when explicitly enabled.

Some pretrained checkpoints may print *new classifier head weights initialized*—this is normal when repurposing a language model for a different number of classes. A CUDA OOM is not an accuracy problem; lower batch size in `fit_transformer` or run a smaller model.

In [13]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeoutError
import time

def _saved_model_status(task,name):
    folder=ROOT/'models'/task/name
    required=[folder/'validated.json',folder/'validation_probabilities.npz']
    if name.startswith('tfidf'): required.append(folder/'model.joblib')
    elif name.startswith('bilstm'): required.extend([folder/'best.pt',folder/'vocab.json'])
    else: required.append(folder/'final'/'config.json')
    missing=[str(path) for path in required if not path.is_file()]
    return ('complete' if not missing else 'incomplete',missing)

if not RUN_MISSING_MODELS:
    print('MODEL TRAINING DISABLED. Existing Drive artifacts are untouched.',flush=True)
    for task in RUN_TASKS:
        for name in RUN_MODELS:
            status,missing=_saved_model_status(task,name)
            if status=='complete': print(f'REUSE {task}/{name}: validated artifacts already exist.')
            else: print(f'SKIP {task}/{name}: incomplete/missing artifacts: {missing}')
    print('To train or resume an incomplete model, explicitly set RUN_MISSING_MODELS=True and rerun this cell.')
else:
    core.set_seed(SEED)
    for task in RUN_TASKS:
        for name in RUN_MODELS:
            print(f'\nSTART {task}/{name} at {time.strftime("%Y-%m-%d %H:%M:%S")}',flush=True)
            status,missing=_saved_model_status(task,name)
            if status=='incomplete': print('Missing before run:',missing,flush=True)
            print('ETA: hardware-dependent; Hugging Face training progress bar estimates remaining time after startup.',flush=True)
            started=time.monotonic()
            with ThreadPoolExecutor(max_workers=1) as executor:
                future=executor.submit(core.fit_one,ROOT,DATA[task],task,name,seed=SEED,
                    epochs_lstm=EPOCHS_BILSTM,epochs_hf=EPOCHS_TRANSFORMER,max_len=MAX_LENGTH)
                while not future.done():
                    try: future.result(timeout=30)
                    except FutureTimeoutError:
                        elapsed=time.monotonic()-started
                        print(f'IN PROGRESS {task}/{name}: elapsed {elapsed/60:.1f} min; still running.',flush=True)
                record=future.result()
            print(f'FINISHED {task}/{name} in {(time.monotonic()-started)/60:.1f} min; '
                  f'macro-F1={record["metrics"]["macro_f1"]:.4f}',flush=True)
        display(core.scoreboard(ROOT,task).round(4))
print('Model cell finished. FINALIZE remains',FINALIZE)

MODEL TRAINING DISABLED. Existing Drive artifacts are untouched.
REUSE cyberbullying7/tfidf_word_char: validated artifacts already exist.
REUSE cyberbullying7/tfidf_svm: validated artifacts already exist.
REUSE cyberbullying7/hing_roberta_mixed: validated artifacts already exist.
REUSE sentiment_raw/tfidf_word_char: validated artifacts already exist.
REUSE sentiment_raw/tfidf_svm: validated artifacts already exist.
REUSE sentiment_raw/hing_roberta_mixed: validated artifacts already exist.
To train or resume an incomplete model, explicitly set RUN_MISSING_MODELS=True and rerun this cell.
Model cell finished. FINALIZE remains False


## 6 — Validation-only comparison and uncertainty

After training at least two candidates, compare their **paired validation predictions** using a bootstrap confidence interval. A small F1 difference with an interval spanning zero is weak evidence for a true improvement.

In [14]:
for task in RUN_TASKS:
    table=core.scoreboard(ROOT,task)
    if table.empty:continue
    display(table.round(4))
    if len(table)>=2:
        result=core.paired_bootstrap_validation(ROOT,task,table.iloc[0].model,table.iloc[1].model,
                                                 repeats=700,seed=SEED)
        print('Paired bootstrap (B minus A):',json.dumps(result,indent=2))

,model,n,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,balanced_accuracy,threshold
0,tfidf_svm,879,0.8203,0.8127,0.8189,0.8169,0.8106,0.8106,0.5
1,hing_roberta_mixed,879,0.8089,0.7958,0.8053,0.8010,0.7983,0.7983,0.5
2,tfidf_word_char,879,0.7873,0.7804,0.7875,0.7831,0.7796,0.7796,0.5
3,twitter_xlmr,879,0.7725,0.7599,0.7684,0.7627,0.7630,0.7630,0.5
4,bilstm_attention,879,0.7190,0.7118,0.7191,0.7203,0.7092,0.7092,0.5
5,muril,879,0.6018,0.5390,0.5584,0.6146,0.5821,0.5821,0.5


Paired bootstrap (B minus A): {
  "task": "cyberbullying7",
  "model_a": "tfidf_svm",
  "model_b": "hing_roberta_mixed",
  "delta_b_minus_a": -0.01693560157316054,
  "bootstrap_95_pct_interval": [
    -0.04588224670107596,
    0.010320305254059354
  ],
  "samples": 879,
  "bootstrap_repeats": 700,
  "scope": "same validation examples; exploratory"
}


,model,n,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,balanced_accuracy,threshold
0,hing_roberta_mixed,1382,0.6650,0.6684,0.6644,0.6681,0.6719,0.6719,0.5
1,twitter_xlmr,1382,0.6425,0.6420,0.6359,0.6392,0.6535,0.6535,0.5
2,muril,1382,0.6360,0.6398,0.6349,0.6388,0.6442,0.6442,0.5
3,tfidf_word_char,1382,0.5912,0.5956,0.5887,0.5929,0.6012,0.6012,0.5
4,tfidf_svm,1382,0.5825,0.5881,0.5832,0.5895,0.5870,0.5870,0.5


Paired bootstrap (B minus A): {
  "task": "sentiment_raw",
  "model_a": "hing_roberta_mixed",
  "model_b": "twitter_xlmr",
  "delta_b_minus_a": -0.026413779861686137,
  "bootstrap_95_pct_interval": [
    -0.04814574254624395,
    -0.00587394950580453
  ],
  "samples": 1382,
  "bootstrap_repeats": 700,
  "scope": "same validation examples; exploratory"
}


## 7 — Optional: validation-only two-model blending

Weights are fit on validation, not test. A blend is retained only when it exceeds the strongest standalone candidate by **at least 1 percentage point** macro-F1. This is a practical guard against microscopic validation gains, not a formal statistical guarantee.

In [15]:
if BUILD_BLEND:
    for task in RUN_TASKS:
        print(task,core.propose_validation_ensemble(ROOT,task,min_gain=.01))
        display(core.scoreboard(ROOT,task).round(4))
else:
    print('BUILD_BLEND=False; no ensemble candidate added.')

BUILD_BLEND=False; no ensemble candidate added.


## 8 — LOCK winner; only then test once

The chosen model and binary threshold are selected on **validation**. After locking, never keep modifying candidates based on test results. For `sentiment`, the public dev holdout has already been exposed in v4; the raw team mirror is not organizer-verified—report it as exploratory historical comparison. Use fresh annotated data for true new external evidence.

In [16]:
if FINALIZE:
    LOCKS=core.select_and_lock(ROOT,DATA,RUN_TASKS)
    print('Frozen selection:',json.dumps({x:LOCKS[x] for x in RUN_TASKS},indent=2))
    for task in RUN_TASKS:
        result=core.test_locked(ROOT,DATA[task],task)
        print(task, 'TEST',json.dumps(result['metrics'],indent=2))
        print('Per class:',pd.DataFrame(result['per_class']).transpose().round(3))
else:
    print('FINALIZE=False: test is not evaluated and the winner is not locked.')

FINALIZE=False: test is not evaluated and the winner is not locked.


## 9 — Robustness and code-mixing script slices (run only after lock)

In [17]:
if FINALIZE:
    for task in RUN_TASKS:
        metrics=core.robustness_locked(ROOT,DATA[task],task)
        print(task,json.dumps(metrics,indent=2))
else:
    print('No test/robustness outputs until explicit lock.')

No test/robustness outputs until explicit lock.


## 10 — Exploratory challenge examples

These examples are deliberately **unlabeled probes**, not reliable ground-truth cases. For publishable numbers, annotate an independent CSV and record annotator disagreement.

In [18]:
CHALLENGE_CASES=[
    ('sarcasm','Wah kya service hai, 3 ghante wait karwaya, amazing 🙃'),
    ('negation','Movie buri nahi thi but ending weak thi'),
    ('roman_variants','yaar ye bahut/bohot achha hai'),
    ('mixed_scripts','कल class hai but I am not ready'),
    ('quoted_insult','Woh gaali de raha tha, aur maine mana kiya'),
    ('soft_negative','This app works but it is very slow'),
    ('neutral','Kal 9 baje meeting hai; link bhej dena')]
if FINALIZE:
    from IPython.display import display
    lock=json.loads((ROOT/'results'/'locked_validation_winners.json').read_text())
    for task in RUN_TASKS:
        loaded=core.load_model(ROOT,task,lock[task]['model'])
        rows=[{'probe':name,'text':t,**core.predict_text(ROOT,task,t,loaded=loaded)} for name,t in CHALLENGE_CASES]
        display(pd.DataFrame(rows)[['probe','text','label','probabilities','review_required']])
else:display(pd.DataFrame(CHALLENGE_CASES,columns=['probe','text']))

,probe,text
0,sarcasm,"Wah kya service hai, 3 ghante wait karwaya, am..."
1,negation,Movie buri nahi thi but ending weak thi
2,roman_variants,yaar ye bahut/bohot achha hai
3,mixed_scripts,कल class hai but I am not ready
4,quoted_insult,"Woh gaali de raha tha, aur maine mana kiya"
5,soft_negative,This app works but it is very slow
6,neutral,Kal 9 baje meeting hai; link bhej dena


## 11 — Independent manually annotated challenge file (optional)

CSV must have `text,label` with **exact class names**, not numeric guesses. At least 20 human-annotated examples, no duplicates and no overlap with the entire experiment. It **fails closed** when overlapping. Better: collect 200–300 across categories with 2 annotators and adjudication.

Example labels for sentiment: `negative`, `neutral`, `positive`. For cyberbullying7: `not_cyberbullying`, `age`, `gender`, `religion`, `mockery`, `abusive`, `offensive`.

In [19]:
if EXTERNAL_CHALLENGE_CSV:
    if not FINALIZE:raise RuntimeError('Lock before evaluating an independent challenge set')
    for task in RUN_TASKS:
        print(json.dumps(core.evaluate_external_challenge(ROOT,task,EXTERNAL_CHALLENGE_CSV),indent=2))
else:print('No externally labeled challenge CSV supplied.')

No externally labeled challenge CSV supplied.


## 12 — Plots and export artifact

CSV/JSON outputs hold **actual measured values**, no fake accuracy metrics. Checkpoints are saved under ROOT but are not included in the small metrics ZIP; archive the full ROOT separately if needed.

In [20]:
import matplotlib.pyplot as plt,zipfile
for task in RUN_TASKS:
    table=core.scoreboard(ROOT,task)
    if table.empty:continue
    fig,ax=plt.subplots(figsize=(10,max(3,0.35*len(table)+1)))
    ax.barh(table.model[::-1],table.macro_f1[::-1])
    ax.set_xlabel('Validation macro-F1');ax.set_xlim(0,1)
    ax.set_title(task+' — validation (NOT leaderboard test)')
    fig.tight_layout();fig.savefig(ROOT/'results'/f'{task}_validation.png',dpi=140);plt.show()
    if FINALIZE:
        res=json.loads((ROOT/'results'/f'{task}_locked_test.json').read_text())
        cm=np.asarray(res['confusion'])
        fig,ax=plt.subplots(figsize=(max(5,len(cm)*.7),max(4,len(cm)*.6)))
        im=ax.imshow(cm,cmap='Blues')
        ax.set_xticks(range(len(core.LABELS[task])),core.LABELS[task],rotation=45,ha='right')
        ax.set_yticks(range(len(core.LABELS[task])),core.LABELS[task])
        ax.set_xlabel('Predicted');ax.set_ylabel('True')
        for i in range(len(cm)):
            for j in range(len(cm)):ax.text(j,i,cm[i,j],ha='center',va='center')
        fig.colorbar(im,ax=ax);fig.tight_layout()
        fig.savefig(ROOT/'results'/f'{task}_confusion.png',dpi=140);plt.show()
metrics_zip=ROOT/'MixBhasha_V5_metrics.zip'
with zipfile.ZipFile(metrics_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for f in (ROOT/'results').glob('*'):
        if f.is_file():z.write(f,arcname=f'results/{f.name}')
print('Metrics:',metrics_zip)
print('All model checkpoints:',ROOT/'models')

Metrics: /content/drive/MyDrive/MixBhasha_V5_seed42/MixBhasha_V5_metrics.zip
All model checkpoints: /content/drive/MyDrive/MixBhasha_V5_seed42/models


## 13 — Honest reporting checklist

- Report **source and exact task taxonomy** for every metric. Do not equate MC-Hinglish cyberbullying, PRISM hate, and HASOC HOF.
- Keep negative/neutral/positive **per-class F1**, especially neutral recall.
- Show TF-IDF, SVM, Bi-LSTM attention, HingRoBERTa-Mixed and Twitter-XLM-R in fair controlled comparisons.
- Show paired bootstrap intervals, robustness under spelling changes, and quality of human-labeled probes.
- Do not call the previously exposed v4 SentiMix dev set a new blind benchmark.
- Cite datasets, model authors, and licenses. The underlying MC-Hinglish1.0 collection is non-commercial; verify model/dataset restrictions before redistribution.
- For app demo: `pip install -r requirements-app.txt && streamlit run app.py` from the project ZIP with `MIXBHASHA_ROOT` set to ROOT. The notebook does not bundle pretrained model weights.

> Claim novel research **only if** the ablation or model improvement holds on independent, annotated evaluations with reproducible results.